# Replicación de Lekkas et al. (2025) en Google Colab
**Deep-Radiomic Fusion for Early Detection of Pancreatic Ductal Adenocarcinoma** · *Applied Sciences*, 15(24), 13024.

Este notebook ejecuta la replicación completa sobre el dataset público **PANORAMA**:

| Paso | Qué hace |
|---|---|
| 0 | Partición por fabricante: desarrollo = Siemens + Philips, prueba externa = Toshiba |
| 1 | Preprocesamiento de las TC (1 mm, −100/600 HU, z-score, recorte), leyendo los zips de Zenodo por streaming |
| 2 | Extracción radiómica con PyRadiomics |
| 3 | **Flujo A:** 16 rasgos estables + gradient boosting |
| 4 | **Flujo B:** CNN 3D CBAM-ResNet-18 |
| 5 | **Flujo C:** fusión deep-radiómica con compuerta squeeze-and-excitation |
| 6 | Evaluación en la prueba externa, comparación con el artículo y Grad-CAM |

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4 o superior).*
Las salidas se guardan en Google Drive (≈ 10–12 GB para el dataset completo), así que una desconexión no hace perder el avance.

> **Recomendación:** ejecuta primero la **prueba piloto** (`PILOTO = True`, unos 30 casos por lote) para verificar que todo funciona; luego cambia a `PILOTO = False`.

In [ ]:
!nvidia-smi -L || echo "Sin GPU: activa una GPU en Entorno de ejecución → Cambiar tipo de entorno"

## Configuración

In [ ]:
PILOTO = True            # True = pocos casos por lote (prueba rápida); False = dataset completo
CASOS_POR_LOTE = 30      # solo se usa en modo piloto
REPO_URL = "https://github.com/20202041-cloud/replicacion-lekkas-2025.git"

from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/replicacion_lekkas"   # aquí quedan los datos procesados y los resultados
import os
os.makedirs(WORK, exist_ok=True)

In [ ]:
%cd /content
!test -d replicacion-lekkas-2025 && (cd replicacion-lekkas-2025 && git pull) || git clone $REPO_URL
%cd /content/replicacion-lekkas-2025
!pip -q install -r requirements.txt remotezip
!pip -q install "pyradiomics @ git+https://github.com/AIM-Harvard/pyradiomics.git"

In [ ]:
# Etiquetas y tabla clínica de PANORAMA (incluye el fabricante de cada TC)
LABELS = f"{WORK}/panorama_labels"
!test -d "$LABELS" && (cd "$LABELS" && git pull) || git clone --depth 1 https://github.com/DIAGNijmegen/panorama_labels "$LABELS"

In [ ]:
import yaml
cfg = yaml.safe_load(open("configs/default.yaml"))
cfg["paths"].update(images_dir="/content/images", labels_repo=LABELS,
                    processed_dir=f"{WORK}/processed", outputs_dir=f"{WORK}/outputs")
cfg["preprocessing"]["n_jobs"] = 2
cfg["radiomics"]["n_jobs"] = 2
cfg["cnn"]["num_workers"] = 2
yaml.safe_dump(cfg, open("configs/colab.yaml", "w"), allow_unicode=True, sort_keys=False)
print("configs/colab.yaml listo")

## Paso 0 · Partición por fabricante

In [ ]:
!python scripts/00_vendor_split.py --config configs/colab.yaml

## Paso 1 · Preprocesamiento (lote por lote, por streaming desde Zenodo)
Cada TC se lee directamente del zip de Zenodo, se preprocesa y se borra. Solo quedan los recortes en Drive.
El paso es **reanudable**: si Colab se desconecta, vuelve a ejecutar esta celda y continúa donde quedó.

In [ ]:
limite = f"--limit {CASOS_POR_LOTE}" if PILOTO else ""
for lote in [1, 2, 3, 4]:
    !python scripts/stream_preprocess.py --config configs/colab.yaml --batch {lote} {limite}

**Plan B (si el streaming falla):** descarga completa de un lote a la vez (necesita ~2× el tamaño del lote en disco).
```python
for lote in [1, 2, 3, 4]:
    !bash scripts/download_panorama.sh {lote} /content/images
    !python scripts/01_build_manifest.py --config configs/colab.yaml --allow-missing-images
    !python scripts/02_preprocess.py --config configs/colab.yaml
    !rm -rf /content/images/batch_{lote}
```

In [ ]:
# Manifiesto final con los casos ya preprocesados
!python scripts/01_build_manifest.py --config configs/colab.yaml --use-processed

## Paso 2 · Radiómica (PyRadiomics)

In [ ]:
!python scripts/03_extract_radiomics.py --config configs/colab.yaml

## Paso 3 · Flujo A: 16 rasgos + gradient boosting

In [ ]:
!python scripts/04_flow_a.py --config configs/colab.yaml

## Paso 4 · Flujo B: CNN 3D CBAM-ResNet-18 (GPU)

In [ ]:
!python scripts/05_train_cnn.py --config configs/colab.yaml

## Paso 5 · Flujo C: fusión deep-radiómica (congelada y extremo a extremo)

In [ ]:
!python scripts/06_train_fusion.py --config configs/colab.yaml --mode frozen
!python scripts/06_train_fusion.py --config configs/colab.yaml --mode end_to_end

## Paso 6 · Evaluación en la prueba externa (Toshiba)

In [ ]:
!python scripts/07_evaluate.py --config configs/colab.yaml --mode frozen
!python scripts/07_evaluate.py --config configs/colab.yaml --mode end_to_end

In [ ]:
from IPython.display import Image, Markdown, display
res = f"{WORK}/outputs/results"
display(Markdown(open(f"{res}/tabla_original_vs_replicacion.md", encoding="utf-8").read()))
display(Image(f"{res}/externa_roc.png", width=420))
display(Image(f"{res}/externa_calibration.png", width=420))

## Guardar los resultados en el repositorio
Copia `tabla_original_vs_replicacion.md`, `metricas.json` y las figuras de `outputs/results` a la carpeta `results/`
del repositorio y súbelos a GitHub. Usa esa tabla para completar la columna **Replicación** del PPT.

In [ ]:
!mkdir -p results && cp -r "$WORK/outputs/results/"* results/ && ls results